# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syedzohairalam123/ML-work1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*
Archetype-to-Action Mapping & Reason Codes:To make model probabilities actionable for editorial teams, model predictions are mapped to concrete, prioritized reason codes based on feature bounds:CTR_FIX_STRIKING: High impression volume in striking distance (Avg Position $\le$ 20) but sub-optimal CTR ($< 1.5\%$). Action: Optimize title tag, meta description, and search result snippet.DECAY_REFRESH: High historic volume coupled with low active days and falling click efficiency. Action: Full content refresh, update outdated facts, and add new section depth.QUICK_WIN: Ranking on Page 2 (Positions 11–20) with healthy baseline CTR. Action: Add internal links from top-performing pages and polish main headings.MAINTAIN: Low decay score and healthy engagement metrics. Action: No immediate intervention required.Action Priority Score: Calculated as $\text{Model Probability} \times \log_{10}(\text{Total Impressions} + 1)$, ensuring high-impact, high-volume opportunities rise to the top of the queue.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import json
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

# Ensure output and figure directories exist
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# Authenticate DuckDB connection
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

con = duckdb.connect()
if HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

# Load performance data for month=2026-03
df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) as total_impressions,
        SUM(gsc_clicks) as total_clicks,
        AVG(gsc_avg_position) as avg_position,
        COUNT(DISTINCT report_date) as active_days,
        ROUND(SUM(gsc_clicks)::FLOAT / NULLIF(SUM(gsc_impressions), 0), 4) as historical_ctr,
        CASE
            WHEN AVG(gsc_avg_position) <= 20 AND (SUM(gsc_clicks)::FLOAT / NULLIF(SUM(gsc_impressions), 0)) < 0.015 THEN 1
            ELSE 0
        END as target_decay
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY 1, 2
    HAVING total_impressions >= 10
""").df().fillna(0)

features = ['total_impressions', 'total_clicks', 'avg_position', 'active_days', 'historical_ctr']
X = df[features]
y = df['target_decay']

# Fit Random Forest baseline classifier
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X, y)

df['decay_probability'] = rf.predict_proba(X)[:, 1]

# Assign Reason Codes and Action Labels
def assign_action(row):
    if row['decay_probability'] > 0.5:
        if row['avg_position'] <= 20 and row['historical_ctr'] < 0.015:
            return 'CTR_FIX_STRIKING', 'Optimize Title & Meta Description'
        else:
            return 'DECAY_REFRESH', 'Full Content Refresh & Depth Update'
    elif 10 < row['avg_position'] <= 20 and row['historical_ctr'] >= 0.015:
        return 'QUICK_WIN', 'Add Internal Links & Heading Polish'
    else:
        return 'MAINTAIN', 'No Action Needed'

actions = df.apply(assign_action, axis=1)
df['reason_code'] = [a[0] for a in actions]
df['recommended_action'] = [a[1] for a in actions]

# Calculate Action Priority Score
df['priority_score'] = (df['decay_probability'] * np.log10(df['total_impressions'] + 1)).round(4)

# Sort Action Queue
ranked_queue = df.sort_values(by='priority_score', ascending=False).reset_index(drop=True)

print(f"Ranked Action Queue generated successfully. Total records: {len(ranked_queue)}")
print(ranked_queue[['content_hash_id', 'priority_score', 'reason_code', 'recommended_action']].head(10).to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Ranked Action Queue generated successfully. Total records: 143206
         content_hash_id  priority_score      reason_code                recommended_action
content_eadb33b5df496f4a          5.7380 CTR_FIX_STRIKING Optimize Title & Meta Description
content_ec2e0346994fb5a5          5.3409 CTR_FIX_STRIKING Optimize Title & Meta Description
content_e8a52cf3d5988c07          5.3264 CTR_FIX_STRIKING Optimize Title & Meta Description
content_0e03de7680314cd5          5.2966 CTR_FIX_STRIKING Optimize Title & Meta Description
content_7172a7fad43f0998          5.2655 CTR_FIX_STRIKING Optimize Title & Meta Description
content_e7b5dd4dff461ad2          5.2604 CTR_FIX_STRIKING Optimize Title & Meta Description
content_8d7d99f109e19aa2          5.2498 CTR_FIX_STRIKING Optimize Title & Meta Description
content_f107e54b10b43725          5.2444 CTR_FIX_STRIKING Optimize Title & Meta Description
content_b99ea6861864dea5          5.2266 CTR_FIX_STRIKING Optimize Title & Meta Description
content_4ffe18

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*
Intended Use:Primary User: Content strategists, SEO managers, and editorial teams.Purpose: Serves as an offline decision-support queue to prioritize weekly content updates and optimization sprints based on measured historical data.Operational Limits:Non-Production Advisory: This playbook is strictly directional and does not execute automated site changes.Data Boundary: Limited to content with $\ge 10$ impressions. Cold-start pages ($< 10$ impressions) and brand-new content are out of scope.External Drivers: Does not account for sudden search engine core algorithm updates, technical site outages, or competitive market shifts.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2 Check: Data Boundary Verification
out_of_scope_count = con.sql(f"""
    SELECT COUNT(*) as count
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    HAVING SUM(gsc_impressions) < 10
""").df()

print(f"Operational Boundary Check: Excluded low-volume/cold-start items (<10 impressions).")

Operational Boundary Check: Excluded low-volume/cold-start items (<10 impressions).


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*
Human Review Checklist:
Before taking action on any flagged URL, a human reviewer must verify:

Intent Alignment: Does the ranking drop reflect shifting search intent that meta updates alone cannot fix?

Technical Sanity: Are canonical tags, indexation settings, and URL redirects intact?

Editorial Value: Is the recommended refresh aligned with current brand guidelines and product offerings?

The No-Go List (Strict Non-Automation Rules):

NO Automated Content Rewrites: AI models must never directly rewrite or publish live article copy without human review.

NO Automated Redirects or Deletions: URL deprecations, 301 redirects, or page unpublishing must never be automated.

NO High-Value Transactional Alters: Conversion-focused checkout or primary landing pages require manual executive sign-off before modification.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 Check: High-Risk Items Flagged for Human Review
high_risk = ranked_queue[
    (ranked_queue['priority_score'] > 2.0) &
    (ranked_queue['reason_code'] == 'DECAY_REFRESH')
]

print(f"Identified {len(high_risk)} high-priority items requiring Mandatory Human Review before editorial action.")

Identified 0 high-priority items requiring Mandatory Human Review before editorial action.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Stale Recommendation Triggers:The action queue risks becoming stale if any of the following triggers are hit:Data Drift: Monthly mean position shift $> 15\%$ or total monthly impression baseline shift $> 25\%$.Concept Drift: Human reviewer rejection rate of recommendations exceeds $30\%$ over two consecutive sprints.Search Algorithm Shift: A major search engine core update invalidates historical CTR-to-position baseline curves.Retrain Schedule: Model retraining must occur monthly upon ingestion of new partition data (month=YYYY-MM).

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Retrain Trigger Diagnostic Check
drift_metrics = {
    'impression_volume_stability': float(df['total_impressions'].std() / df['total_impressions'].mean()),
    'decay_flag_ratio': float(df['target_decay'].mean()),
    'acceptance_threshold_min': 0.70
}

print("Monitoring & Retrain Triggers defined:")
print(json.dumps(drift_metrics, indent=2))

Monitoring & Retrain Triggers defined:
{
  "impression_volume_stability": 3.049171917712529,
  "decay_flag_ratio": 0.6985740820915325,
  "acceptance_threshold_min": 0.7
}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*
xporting the prioritized action queue CSV to work/outputs/action_queue.csv, summary key-performance metrics JSON to work/outputs/action_metrics.json, and summary distribution figure to work/figures/action_distribution.png for inclusion in the final research paper.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Export Ranked Action Queue CSV
queue_export_path = 'work/outputs/action_queue.csv'
ranked_queue[['client_hash_id', 'content_hash_id', 'priority_score', 'decay_probability',
              'reason_code', 'recommended_action', 'total_impressions', 'avg_position', 'historical_ctr']
].to_csv(queue_export_path, index=False)

# 2. Export Metrics JSON
metrics = {
    'total_evaluated_urls': int(len(ranked_queue)),
    'decay_refresh_count': int((ranked_queue['reason_code'] == 'DECAY_REFRESH').sum()),
    'ctr_fix_striking_count': int((ranked_queue['reason_code'] == 'CTR_FIX_STRIKING').sum()),
    'quick_win_count': int((ranked_queue['reason_code'] == 'QUICK_WIN').sum()),
    'maintain_count': int((ranked_queue['reason_code'] == 'MAINTAIN').sum()),
    'top_priority_score_max': float(ranked_queue['priority_score'].max())
}

with open('work/outputs/action_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

# 3. Export Summary Action Distribution Plot
plt.figure(figsize=(8, 4))
ranked_queue['reason_code'].value_counts().plot(kind='bar', color='#1f77b4', edgecolor='black')
plt.title('Content Action Playbook Queue Distribution')
plt.xlabel('Reason Code')
plt.ylabel('URL Count')
plt.tight_layout()
plt.savefig('work/figures/action_distribution.png', dpi=300)
plt.close()

print(f"Successfully exported:")
print(f" - CSV: {queue_export_path}")
print(f" - JSON: work/outputs/action_metrics.json")
print(f" - Figure: work/figures/action_distribution.png")

Successfully exported:
 - CSV: work/outputs/action_queue.csv
 - JSON: work/outputs/action_metrics.json
 - Figure: work/figures/action_distribution.png


## Self-check

Before you submit, confirm each line honestly:

[x] Every section above is filled — markdown thinking AND the code that backs it

[x] The notebook runs top to bottom with no errors (Runtime → Run all)

[x] No client names, URLs, or private queries anywhere

[x] My claims use careful words: observed, measured, directional, decision-support

[x] Committed to my repo under work/notebooks/w07_action_playbook.ipynb — then submit your repo URL on the card. Done.